# Entrenamiento en Google Colab — CAMUS Segmentación

**Antes de correr:** Menú → Runtime → Change runtime type → **GPU (T4)**

Este notebook entrena los dos modelos y genera los resultados comparativos.

## 0. Setup: GPU, Drive y repo

In [ ]:
# Verificar GPU
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Montar Google Drive (los checkpoints se guardan aquí para no perderlos)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/camus_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints se guardarán en: {DRIVE_DIR}')

In [ ]:
# Clonar el repo (reemplazar con tu URL)
REPO_URL = 'https://github.com/TU_USUARIO/camus-lv-segmentation.git'
!git clone {REPO_URL}
%cd camus-lv-segmentation
!pip install -r requirements.txt -q

## 1. Descargar dataset CAMUS desde Kaggle

In [ ]:
# Configurar Kaggle API
# Instrucciones: https://www.kaggle.com/docs/api
# 1. Ir a kaggle.com → Account → API → Create New Token
# 2. Se descarga kaggle.json
# 3. En Colab: Secrets (🔑) → agregar KAGGLE_API_TOKEN con el contenido del json

from google.colab import userdata
import os, json

kaggle_token = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)
!chmod 600 /root/.kaggle/kaggle.json

# Descargar y descomprimir
!kaggle datasets download -d shoybhasan/camus-human-heart-data -q
!unzip -q camus-human-heart-data.zip -d dataset_raw
!rm camus-human-heart-data.zip
print('Dataset descargado.')

In [ ]:
# Descomprimir el archivo interno
import zipfile, os
inner = 'dataset_raw/download'
if os.path.exists(inner):
    with zipfile.ZipFile(inner, 'r') as z:
        z.extractall('datos_corazon')
    print('Dataset listo en: datos_corazon/')
    !ls datos_corazon/ | head -5
else:
    print('Buscar el archivo comprimido interno:')
    !find dataset_raw -name '*.zip' -o -name 'download'

## 2. EDA rápido

In [ ]:
import sys
sys.path.insert(0, 'src')
from dataset import prepare_dataset

loaders = prepare_dataset(
    data_root='datos_corazon',
    batch_size=8,
    num_workers=2,
)

# Mostrar un batch de ejemplo
import matplotlib.pyplot as plt
import numpy as np

imgs, masks = next(iter(loaders['train']))
print(f'Batch imágenes: {imgs.shape}  dtype={imgs.dtype}')
print(f'Batch máscaras: {masks.shape}  clases={masks.unique().tolist()}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(imgs[i, 0].numpy(), cmap='gray')
    axes[0, i].set_title(f'Imagen {i+1}'); axes[0, i].axis('off')
    axes[1, i].imshow(masks[i].numpy(), cmap='jet', vmin=0, vmax=3)
    axes[1, i].set_title(f'Máscara {i+1}'); axes[1, i].axis('off')
plt.suptitle('Muestra del Dataset — Train Set', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Entrenar Modelo 1: U-Net + ResNet34

In [ ]:
from train import train, DEFAULT_CONFIG

config_unet = {**DEFAULT_CONFIG,
    'model':      'unet',
    'encoder':    'resnet34',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  'datos_corazon',
    'save_dir':   DRIVE_DIR,    # guardar en Drive
    'device':     'cuda',
}

history_unet = train(config_unet)

## 4. Entrenar Modelo 2: Attention U-Net + EfficientNet-B0

In [ ]:
config_attn = {**DEFAULT_CONFIG,
    'model':      'attention-unet',
    'encoder':    'efficientnet-b0',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  'datos_corazon',
    'save_dir':   DRIVE_DIR,
    'device':     'cuda',
}

history_attn = train(config_attn)

## 5. Curvas de entrenamiento

In [ ]:
from evaluate import plot_training_history

plot_training_history(history_unet,  'U-Net + ResNet34',              'unet_history.png')
plot_training_history(history_attn,  'Attention U-Net + EfficientNet-B0', 'attn_history.png')

## 6. Evaluación en Test Set

In [ ]:
import torch
from models   import get_model
from losses   import CombinedLoss
from evaluate import evaluate, print_metrics_table, visualize_predictions, compare_models

device  = torch.device('cuda')
loss_fn = CombinedLoss()
results = {}

# Evaluar U-Net
ckpt1 = torch.load(f'{DRIVE_DIR}/best_unet_resnet34.pth', map_location=device)
m1    = get_model('unet', encoder_name='resnet34').to(device)
m1.load_state_dict(ckpt1['model_state'])
r1    = evaluate(m1, loaders['test'], loss_fn, device, 'U-Net')
print_metrics_table(r1, 'U-Net + ResNet34')
results['U-Net + ResNet34'] = r1

# Visualizaciones U-Net
visualize_predictions(m1, loaders['test'], device,
                      n_samples=6, save_path='predictions_unet.png',
                      model_name='U-Net + ResNet34')

# Evaluar Attention U-Net
ckpt2 = torch.load(f'{DRIVE_DIR}/best_attention-unet_efficientnet-b0.pth', map_location=device)
m2    = get_model('attention-unet', encoder_name='efficientnet-b0').to(device)
m2.load_state_dict(ckpt2['model_state'])
r2    = evaluate(m2, loaders['test'], loss_fn, device, 'Attention U-Net')
print_metrics_table(r2, 'Attention U-Net + EfficientNet-B0')
results['Attention U-Net + EfficientNet-B0'] = r2

# Visualizaciones Attention U-Net
visualize_predictions(m2, loaders['test'], device,
                      n_samples=6, save_path='predictions_attn.png',
                      model_name='Attention U-Net + EfficientNet-B0')

## 7. Comparativa final entre modelos

In [ ]:
compare_models(results, save_path='comparativa_modelos.png')

# Tabla resumen
print('\n' + '='*65)
print(f'{"Modelo":<40} {"LV":>6} {"MYO":>6} {"LA":>6} {"Mean":>6}')
print('='*65)
for name, m in results.items():
    print(f'{name:<40} {m["dice_lv"]:>6.3f} {m["dice_myo"]:>6.3f} {m["dice_la"]:>6.3f} {m["mean_dice_cardiac"]:>6.3f}')
print(f'{"Referencia (Leclerc 2019)":<40} {0.94:>6.3f} {0.85:>6.3f} {0.89:>6.3f} {0.89:>6.3f}')
print('='*65)

## 8. Descargar resultados

Copiar los archivos generados a Drive para no perderlos.

In [ ]:
import shutil
archivos = ['predictions_unet.png', 'predictions_attn.png',
            'comparativa_modelos.png', 'unet_history.png', 'attn_history.png']

for f in archivos:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print(f'Copiado: {f} → {DRIVE_DIR}')

print('\n✓ Todos los resultados guardados en Google Drive')